In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy
import pickle

movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

In [2]:
#Prepare data for surprise library
# Surprise library needs data in a specific format
# Reader tells it: ratings are on a scale of 0.5 to 5.0
reader = Reader(rating_scale=(0.5, 5.0))

# Load data into Surprise's format
data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

print("Data loaded successfully!")
print(f"Total ratings: {len(ratings)}")

Data loaded successfully!
Total ratings: 100836


In [3]:
# Split: 80% training, 20% testing
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f"Training ratings: {trainset.n_ratings}")
print(f"Test ratings: {len(testset)}")

Training ratings: 80668
Test ratings: 20168


In [4]:
#Train SVD model
# SVD model with tuned parameters
svd_model = SVD(
    n_factors=100,    # number of hidden features to discover
    n_epochs=20,      # how many times to iterate over training data
    lr_all=0.005,     # learning rate
    reg_all=0.02,     # regularization to prevent overfitting
    random_state=42
)

# Train the model
svd_model.fit(trainset)
print("Model trained!")

Model trained!


In [5]:
#evaluate the model
# Make predictions on test set
predictions = svd_model.test(testset)

# RMSE — Root Mean Square Error
# Lower is better. On a 0.5-5.0 scale, below 1.0 is good
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(f"\nRMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"\nInterpretation: On average, our predictions are off by {rmse:.2f} stars")

RMSE: 0.8807
MAE:  0.6766

RMSE: 0.8807
MAE: 0.6766

Interpretation: On average, our predictions are off by 0.88 stars


In [6]:
#Cross validation
# Cross validation tests model on 5 different splits
# This proves the model is consistent, not just lucky on one split
cv_results = cross_validate(
    SVD(n_factors=100, n_epochs=20, random_state=42),
    data,
    measures=['RMSE', 'MAE'],
    cv=5,
    verbose=True
)

print(f"\nAverage RMSE across 5 folds: {cv_results['test_rmse'].mean():.4f}")
print(f"Average MAE across 5 folds: {cv_results['test_mae'].mean():.4f}")

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8752  0.8583  0.8777  0.8752  0.8733  0.8719  0.0070  
MAE (testset)     0.6706  0.6635  0.6710  0.6736  0.6713  0.6700  0.0034  
Fit time          1.55    1.53    1.53    1.47    1.54    1.52    0.03    
Test time         0.19    0.13    0.23    0.28    0.11    0.19    0.06    

Average RMSE across 5 folds: 0.8719
Average MAE across 5 folds: 0.6700


In [7]:
#Get recommendations for the user
def get_collaborative_recommendations(user_id, top_n=10):
    """
    Given a user_id, predict ratings for all movies they haven't seen
    and return top N recommendations.
    """
    # Movies this user has already rated
    rated_movies = ratings[ratings['userId'] == user_id]['movieId'].tolist()
    
    # All movies in dataset
    all_movie_ids = movies['movieId'].tolist()
    
    # Movies the user has NOT seen
    unseen_movies = [m for m in all_movie_ids if m not in rated_movies]
    
    # Predict rating for each unseen movie
    predictions_list = []
    for movie_id in unseen_movies:
        pred = svd_model.predict(user_id, movie_id)
        predictions_list.append({
            'movieId': movie_id,
            'predicted_rating': round(pred.est, 3)
        })
    
    # Convert to dataframe and sort
    pred_df = pd.DataFrame(predictions_list)
    pred_df = pred_df.sort_values('predicted_rating', ascending=False)
    
    # Merge with movie titles
    result = pred_df.merge(movies, on='movieId')
    
    return result[['title', 'genres', 'predicted_rating']].head(top_n)

# Test with User 1
print("=== Top 10 Recommendations for User 1 ===")
print(get_collaborative_recommendations(user_id=1))

print("\n=== Top 10 Recommendations for User 42 ===")
print(get_collaborative_recommendations(user_id=42))

=== Top 10 Recommendations for User 1 ===
                                               title  \
0                          North by Northwest (1959)   
1                         Lost in Translation (2003)   
2        Seven Samurai (Shichinin no samurai) (1954)   
3                               Departed, The (2006)   
4  Lord of the Rings: The Return of the King, The...   
5                                Blade Runner (1982)   
6                       Boot, Das (Boat, The) (1981)   
7  Dr. Strangelove or: How I Learned to Stop Worr...   
8                                  Casablanca (1942)   
9    Grand Day Out with Wallace and Gromit, A (1989)   

                                       genres  predicted_rating  
0   Action|Adventure|Mystery|Romance|Thriller               5.0  
1                        Comedy|Drama|Romance               5.0  
2                      Action|Adventure|Drama               5.0  
3                        Crime|Drama|Thriller               5.0  
4          

In [8]:
# Let's see what User 1 already rated highly
# This helps us verify recommendations make sense
user1_ratings = ratings[ratings['userId'] == 1].merge(movies, on='movieId')
user1_ratings = user1_ratings.sort_values('rating', ascending=False)

print("=== Movies User 1 rated highly ===")
print(user1_ratings[['title', 'genres', 'rating']].head(10))

=== Movies User 1 rated highly ===
                                           title  \
231                 M*A*S*H (a.k.a. MASH) (1970)   
185                             Excalibur (1981)   
89     Indiana Jones and the Last Crusade (1989)   
90                   Pink Floyd: The Wall (1982)   
190                 From Russia with Love (1963)   
189                            Goldfinger (1964)   
188                      Dirty Dozen, The (1967)   
186                    Gulliver's Travels (1939)   
184                       American Beauty (1999)   
179  South Park: Bigger, Longer and Uncut (1999)   

                           genres  rating  
231              Comedy|Drama|War     5.0  
185             Adventure|Fantasy     5.0  
89               Action|Adventure     5.0  
90                  Drama|Musical     5.0  
190     Action|Adventure|Thriller     5.0  
189     Action|Adventure|Thriller     5.0  
188              Action|Drama|War     5.0  
186  Adventure|Animation|Children     5.

In [9]:
# Save SVD model for use in the app later
with open('../data/svd_model.pkl', 'wb') as f:
    pickle.dump(svd_model, f)

print("SVD model saved!")

SVD model saved!
